# Multi-Agent Orchestration Across Multiple LLM Providers

This notebook demonstrates how to build a **multi-agent sales workflow** using the OpenAI Agents SDK while combining models from several providers.

The workflow uses:

- **Gemini** for one sales-writing agent
- **Kimi through OpenRouter** for another sales-writing agent and the manager
- **GPT-OSS through Groq** for a third sales-writing agent
- A **Sales Manager agent** that coordinates the other agents
- An email tool that sends only the best draft

## Workflow Architecture

```mermaid
flowchart TD
    U[User Task] --> M[Sales Manager]
    M --> A1[Gemini Sales Agent]
    M --> A2[Kimi Sales Agent]
    M --> A3[GPT-OSS Sales Agent]
    A1 --> M
    A2 --> M
    A3 --> M
    M --> E[Email Sending Tool]
```

The main objective is to show how one manager agent can delegate the same task to multiple specialist agents, compare their responses, and execute a final action.

In [1]:
from dotenv import load_dotenv
from openai import AsyncOpenAI
from agents import Agent, Runner, trace, function_tool, OpenAIChatCompletionsModel, output_guardrail, GuardrailFunctionOutput
import os
from pydantic import BaseModel, Field
from email.message import EmailMessage
import smtplib
import requests
load_dotenv(override=True)

True

## 1. Configure Email Delivery

The notebook reads the SMTP configuration from environment variables.

Keeping credentials in a `.env` file prevents sensitive information from being written directly into the notebook or committed to GitHub.

In [2]:




EMAIL_ADDRESS = os.getenv("EMAIL_ADDRESS")
EMAIL_SMTP_SERVER = os.getenv("EMAIL_SMTP_SERVER")
EMAIL_APP_PASSWORD = os.getenv("EMAIL_APP_PASSWORD")

def send_email(subject, text_body, html_body):
    msg = EmailMessage()
    msg["From"] = EMAIL_ADDRESS
    msg["To"] = EMAIL_ADDRESS
    msg["Subject"] = subject
    msg.set_content(text_body)
    msg.add_alternative(html_body, subtype="html")

    with smtplib.SMTP(EMAIL_SMTP_SERVER, 587) as server:
        server.starttls()
        server.login(EMAIL_ADDRESS, EMAIL_APP_PASSWORD)
        server.send_message(msg)


pushover_user = os.getenv("PUSHOVER_USER")
pushover_token = os.getenv("PUSHOVER_TOKEN")
pushover_url = "https://api.pushover.net/1/messages.json"

def push(message):
    print(f"Push: {message}")
    payload = {"user": pushover_user, "token": pushover_token, "message": message}
    requests.post(pushover_url, data=payload)

## 2. Load and Validate API Keys

The workflow uses API keys from multiple providers.

The checks below confirm whether each key was loaded without printing the complete secret. An OpenAI API key is optional when OpenAI models and OpenAI-hosted tracing are not being used.

In [3]:
openai_api_key = os.getenv('OPENAI_API_KEY')
google_api_key = os.getenv('GOOGLE_API_KEY')
openrouter_api_key = os.getenv('OPEN_ROUTER_KEY')
groq_api_key = os.getenv('GROQ_API_KEY')


if google_api_key:
    print(f"Google API Key exists and begins {google_api_key[:2]}")
else:
    print("Google API Key not set (and this is optional)")

if  openrouter_api_key:
    print(f"OpenRouter API Key exists and begins {openrouter_api_key[:6]}")
else:
    print("OpenRouter API Key not set (and this is optional)")

if groq_api_key:
    print(f"Groq API Key exists and begins {groq_api_key[:4]}")
else:
    print("Groq API Key not set (and this is optional)")

Google API Key exists and begins AI
OpenRouter API Key exists and begins sk-or-
Groq API Key exists and begins gsk_


## 3. Define the Sales-Agent Instructions

All three writer agents receive the same core instructions.

Because the agents use different underlying models, they can still produce distinct writing styles and sales approaches from the same prompt.

In [4]:
instructions = """
You are a sales agent working for DEEPLABAI, 
a company that provides a SaaS tool for ensuring SOC2 compliance and preparing for audits, powered by AI.
You write compelling sales emails that are likely to get a response.
"""

## 4. Define Provider Base URLs

Gemini, OpenRouter, and Groq expose OpenAI-compatible API endpoints.

These base URLs allow the OpenAI Python client and the Agents SDK model adapter to communicate with each provider through a consistent interface.

In [5]:
GEMINI_BASE_URL = "https://generativelanguage.googleapis.com/v1beta/openai/"
OPENROUTER_BASE_URL = "https://openrouter.ai/api/v1"
GROQ_BASE_URL = "https://api.groq.com/openai/v1"

## 5. Create Provider Clients

A separate asynchronous client is initialized for every provider.

Each client uses its provider-specific base URL and API key.

In [6]:
gemini_client = AsyncOpenAI(base_url=GEMINI_BASE_URL, api_key=google_api_key)
openrouter_client = AsyncOpenAI(base_url=OPENROUTER_BASE_URL, api_key=openrouter_api_key)
groq_client = AsyncOpenAI(base_url=GROQ_BASE_URL, api_key=groq_api_key)

## 6. Configure the Language Models

The provider clients are wrapped with `OpenAIChatCompletionsModel`.

This lets the Agents SDK use models hosted outside OpenAI while keeping a common agent interface.

In [7]:
gemini_model = OpenAIChatCompletionsModel(model="gemini-3.1-flash-lite", openai_client=gemini_client)
kimi_model = OpenAIChatCompletionsModel(model="moonshotai/kimi-k2.6", openai_client=openrouter_client)
oss_model = OpenAIChatCompletionsModel(model="openai/gpt-oss-120b", openai_client=groq_client)

## 7. Create the Specialist Sales Agents

Three independent sales agents are created.

Each agent receives the same role and instructions, but each one is powered by a different language model. This creates several candidate drafts for the manager to evaluate.

In [8]:
sales_agent1 = Agent(name="Gemini Sales Agent", instructions=instructions, model=gemini_model)
sales_agent2 = Agent(name="Kimi Sales Agent", instructions=instructions, model=kimi_model)
sales_agent3 = Agent(name="GPT-OSS Sales Agent",instructions=instructions, model=oss_model)
  

## 8. Convert the Agents into Tools

The specialist agents are exposed as tools using `as_tool()`.

This is a hierarchical multi-agent pattern: the manager does not manually call each model API. Instead, it invokes the specialist agents as tools during its own reasoning process.

In [9]:
description = "Use this tool to write a sales email. In the input, just instruct it to write a sales email."

tool1 = sales_agent1.as_tool(tool_name="sales_agent1", tool_description=description)
tool2 = sales_agent2.as_tool(tool_name="sales_agent2", tool_description=description)
tool3 = sales_agent3.as_tool(tool_name="sales_agent3", tool_description=description)

## 9. Configure the Email Execution Function

`send_message()` controls whether the final result is sent through email or displayed through an alternative output function.

The `USE_EMAIL` flag is useful during development because it can prevent accidental email delivery while testing.

In [10]:


USE_EMAIL = True

def send_message(subject, text_body, html_body):
    if USE_EMAIL:
        send_email(subject, text_body, html_body)
    else:
        push(f"Subject: {subject}\n\n{text_body}")

## 10. Test the Email Configuration

This cell sends a simple test message.

Run it only when the SMTP credentials are configured correctly and you intentionally want to verify email delivery.

In [11]:
send_message("Yet another test", "Hooray!", "<html><body><h1>Hooray!</h1></body></html>")

## 11. Expose Email Delivery as a Tool

The email function is wrapped with `@function_tool`.

This allows the Sales Manager agent to execute the final real-world action after selecting the strongest draft.

In [12]:
@function_tool
def send_email_tool(subject: str, text_body: str, html_body: str) -> str:
    """
    Send out an email with the given subject and body to all sales prospects
    
    Args:
        subject: The subject of the email
        text_body: The body of the email as plain text
        html_body: The HTML body of the email
    """
    send_message(subject, text_body, html_body)
    return "Email sent successfully"

## 12. Register the Available Tools

The manager receives four tools:

1. Gemini Sales Agent
2. Kimi Sales Agent
3. GPT-OSS Sales Agent
4. Email Sending Tool

The first three produce candidate drafts, while the last tool performs the final action.

In [13]:
tools = [tool1, tool2, tool3, send_email_tool]

## 13. Define the Sales Manager and Task

The Sales Manager is responsible for the complete orchestration process.

It must:

1. Call all three writer agents
2. Wait for all three drafts
3. Compare the drafts
4. Select the strongest email
5. Send exactly one final email

The task prompt describes the required sequence of actions.

In [14]:
instructions = """
You are a Sales Manager at DEEPLABAI. Your goal is to find the single best cold sales email using the sales_writer tools.
"""

task = """
Follow these steps:

1. Generate Drafts: Use each of the three sales_email_writer tools to generate different email drafts.
Just instruct each to write a sales email; no further details are needed.
Do not proceed until all three drafts are ready, one from each tool.
 
2. Evaluate and Select: Review the drafts and choose the single best email using your judgment of which one is most effective.
 
3. Use your tool to send the best email (and only the best email) to the user. Only send 1 email.
"""

sales_manager = Agent(name="Sales Manager", instructions=instructions, model=kimi_model, tools=tools)

## 14. Run the Multi-Agent Workflow

The workflow is executed with `Runner.run()`.

The `trace()` context groups the agent activity under one workflow name. OpenAI-hosted trace visualization requires an OpenAI API key, but the workflow itself can still run with Gemini, OpenRouter, and Groq models without one.

In [15]:
with trace("Sales Manager across different models"):
    result = await Runner.run(sales_manager, task)
print(result.final_output)

OPENAI_API_KEY is not set, skipping trace export
OPENAI_API_KEY is not set, skipping trace export
OPENAI_API_KEY is not set, skipping trace export
OPENAI_API_KEY is not set, skipping trace export
OPENAI_API_KEY is not set, skipping trace export


All done! I generated three different cold sales emails, evaluated them, and selected **Draft 2** as the single best option because it:

- Opens with immediate empathy for the prospect's pain  
- Delivers a clear, compelling value proposition (70% time savings)  
- Uses concise, punchy language ideal for cold outreach  
- Includes a low-friction call-to-action (15-minute call)

I’ve sent the winning email to our sales prospects with the subject line **"SOC2 without the spreadsheet marathon?"**


OPENAI_API_KEY is not set, skipping trace export


# Summary

In this notebook, you built a cross-provider multi-agent system with the OpenAI Agents SDK.

The workflow demonstrates:

- Connecting to multiple OpenAI-compatible providers
- Creating specialist agents with different LLMs
- Converting agents into callable tools
- Building a manager-orchestrator agent
- Comparing multiple generated outputs
- Executing a real action through a function tool

This architecture can be adapted for research, content generation, customer support, code review, document analysis, and other workflows where several specialist agents contribute to one final decision.